In [0]:
import urllib.request
import os
import shutil
from datetime import date, datetime, timezone
from dateutil.relativedelta import relativedelta

# get two months ago relative to today's date and format it
two_months_ago = date.today() - relativedelta(months=2)
formatted_date = two_months_ago.strftime("%Y-%m")

# create base-save path
dir_path = f"/Volumes/workspace/00_landing/data_sources/nyctaxi_yellow/{formatted_date}"

# extend with base-save with file
local_path = f"{dir_path}/yellow_tripdata_{formatted_date}.parquet"

try:
    # check if file exists
    dbutils.fs.ls(local_path)

    # file exists then set continue to NO
    dbutils.jobs.taskValues.set(key="continue_downstream", value="no")
    print("File already downloaded, aborting downstream tasks")
except:
    try:

        # create url
        url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{formatted_date}.parquet"

        # open a connection to remote url
        response = urllib.request.urlopen(url)

        # create the local directory for this date's data
        os.makedirs(dir_path, exist_ok=True)

        # save data
        with open(local_path, 'wb') as f:
            shutil.copyfileobj(response, f)

        # set contine to yes
        dbutils.jobs.taskValues.set(key="continue_downstream", value="yes")

    except Exception as e:
        dbutils.jobs.taskValues.set(key="continue_downstream", value="no")
        print(f"File download failed {e}")